# 01 — Download audio from Xeno-canto

This notebook demonstrates how to use `src/download.py` to download `.mp3` audio recordings from the [Xeno-canto](https://xeno-canto.org/) API and organise them under `data/raw/<species>/`.

**Pipeline:**
1. Environment setup (Colab / local)
2. API key and species list configuration
3. Search recordings
4. Download audio
5. Verify downloaded files

## 1. Environment setup

The cell below automatically detects Google Colab and, if running there, clones the repository and installs all dependencies.

In [ ]:
import sys
import os
import importlib

# Detect Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    # Persistent data directory on Drive — survives across sessions
    DRIVE_DATA_ROOT = '/content/drive/MyDrive/bird-acoustics-classifier'
    os.makedirs(f'{DRIVE_DATA_ROOT}/data/raw', exist_ok=True)

    repo_dir = '/content/bird-acoustics-classifier'
    if not os.path.exists(repo_dir):
        !git clone https://github.com/danort92/bird-acoustics-classifier.git {repo_dir}
    else:
        !git -C {repo_dir} pull origin claude/setup-project-structure-lVRTH

    if repo_dir not in sys.path:
        sys.path.insert(0, repo_dir)
    os.chdir(repo_dir)
    !pip install -q -r requirements.txt

    if 'src.download' in sys.modules:
        importlib.reload(sys.modules['src.download'])

    print('Colab setup complete.')
    print(f'Drive data root: {DRIVE_DATA_ROOT}')
else:
    DRIVE_DATA_ROOT = None
    project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
    if project_root not in sys.path:
        sys.path.insert(0, project_root)
    os.chdir(project_root)
    print(f'Working directory: {os.getcwd()}')


## 2. Configuration

Set the **Xeno-canto API key** and the **species list** to download.

> **API key — required since October 2025.**
> The public API v2 has been shut down; API v3 requires authentication.
> Get a free key by registering at [xeno-canto.org](https://xeno-canto.org)
> and visiting your [Account page](https://xeno-canto.org/article/854).
> Set the key as `XENO_CANTO_API_KEY` or enter it when prompted below.

The species below are characteristic of the **Alpine zone (Italy / Austria / Switzerland)**,
covering a range of habitats from subalpine forests and meadows to rocky peaks.

In [ ]:
# Set your API key (required for Xeno-canto API v3)
# Option A — environment variable (recommended, key never appears in notebook output)
# os.environ['XENO_CANTO_API_KEY'] = 'your_key_here'

# Option B — enter interactively when the downloader is initialised (see cell 3)

# Species characteristic of the Alpine zone (Italy / Austria / Switzerland)
SPECIES = [
    "Turdus torquatus",        # Ring ouzel              — rocky slopes, high-altitude forests
    "Phoenicurus ochruros",    # Black redstart           — rocky terrain, mountain villages
    "Prunella collaris",       # Alpine accentor          — high rocky areas above treeline
    "Pyrrhocorax graculus",    # Yellow-billed chough     — alpine cliffs and glaciers
    "Pyrrhocorax pyrrhocorax", # Red-billed chough        — alpine meadows and cliffs
    "Tichodroma muraria",      # Wallcreeper              — vertical rock faces
    "Anthus spinoletta",       # Water pipit              — alpine meadows and streams
    "Montifringilla nivalis",  # White-winged snowfinch   — above treeline, snowfields
    "Lagopus muta",            # Rock ptarmigan           — high alpine tundra
    "Dryocopus martius",       # Black woodpecker         — subalpine conifer forests
    "Tetrao urogallus",        # Western capercaillie     — old-growth conifer forests
    "Picoides tridactylus",    # Three-toed woodpecker    — spruce forests
    "Loxia curvirostra",       # Common crossbill         — conifer forests
    "Nucifraga caryocatactes", # Spotted nutcracker       — mountain conifer forests
    "Regulus ignicapilla",     # Firecrest                — mixed mountain forests
    "Cinclus cinclus",         # White-throated dipper    — alpine streams and torrents
    "Ficedula albicollis",     # Collared flycatcher      — deciduous mountain forests
    "Saxicola rubetra",        # Whinchat                 — subalpine meadows
    "Emberiza cia",            # Rock bunting             — rocky slopes with sparse vegetation
    "Gypaetus barbatus",       # Bearded vulture          — high alpine cliffs (reintroduced)
]

MAX_PER_SPECIES = 30   # Maximum number of files per species

# Quality filter: "A" = only top-grade recordings; any other value = all grades accepted.
# Xeno-canto API does not support OR across quality values, so "all grades" is
# the practical way to maximise training data while keeping correct species labels.
QUALITY = "all"

OUTPUT_DIR = "data/raw"

print(f"Selected species ({len(SPECIES)}) — Alpine zone:")
for s in SPECIES:
    print(f"  {s}")
print(f"\nQuality filter: {QUALITY}")

## 3. Initialise the downloader

In [ ]:
from src.download import XenoCantoDownloader

# The constructor automatically reads XENO_CANTO_API_KEY
# or prompts interactively if not set.
downloader = XenoCantoDownloader(
    output_dir=OUTPUT_DIR,
    request_delay=0.5,  # pause between requests (seconds)
)
print("Downloader initialised.")

## 4. Search recordings (preview — no download)

Before downloading, check how many recordings are available for each species.

In [ ]:
import pandas as pd

preview_rows = []

for species in SPECIES:
    recordings = downloader.search_species(species, quality=QUALITY, max_results=MAX_PER_SPECIES)
    for r in recordings[:3]:
        preview_rows.append({
            "species": species,
            "id": r.get("id"),
            "country": r.get("cnt"),
            "quality": r.get("q"),
            "length": r.get("length"),
            "file_url": r.get("file"),
        })
    print(f"{species}: {len(recordings)} recordings found")

df_preview = pd.DataFrame(preview_rows)
df_preview

## 5. Download audio

Download all recordings for the configured species and save them to `data/raw/<species>/`.

In [ ]:
results = downloader.download_species(
    species_list=SPECIES,
    max_per_species=MAX_PER_SPECIES,
    quality=QUALITY,
)

print("\n=== Download summary ===")
for species, paths in results.items():
    print(f"  {species}: {len(paths)} files downloaded")

## 6. Verify downloaded files

In [ ]:
from pathlib import Path

downloaded = downloader.list_downloaded()

summary_rows = []
for species_dir, files in downloaded.items():
    total_mb = sum(f.stat().st_size for f in files) / 1_048_576
    summary_rows.append({
        "species_dir": species_dir,
        "n_files": len(files),
        "total_MB": round(total_mb, 2),
    })

df_summary = pd.DataFrame(summary_rows)
print(df_summary.to_string(index=False))
print(f"\nTotal files: {df_summary['n_files'].sum()}")
print(f"Total size: {df_summary['total_MB'].sum():.2f} MB")

## 7. Resulting directory structure

Print the directory tree after the download.

In [ ]:
def print_tree(root: Path, max_files: int = 5) -> None:
    """Print a compact directory tree."""
    print(f"{root}/")
    for species_dir in sorted(root.iterdir()):
        if not species_dir.is_dir():
            continue
        files = sorted(species_dir.glob('*.mp3'))
        print(f"  {species_dir.name}/  ({len(files)} files)")
        for f in files[:max_files]:
            print(f"    {f.name}")
        if len(files) > max_files:
            print(f"    ... and {len(files) - max_files} more files")

print_tree(Path(OUTPUT_DIR))

---
**Next step:** `02_preprocessing.ipynb` — convert audio to mel spectrograms